# Evaluate Models — number+percent (all_open) features, NEW vs OLD

Compares **`model_npo_new`** vs **`model_npo_old`** (from
`Build_Model_NumberPercentOpen_{New,Old}.ipynb`): same 288 frozen features
(128 `number_of_DQ*` + 160 `percent*`, `all_open*` groups), same applicants —
the only difference is the aggregator behavior that produced the data
(NEW = denominator fix + experian placeholder change; OLD = shipping).

The number features are what the experian placeholder change moves, so the
slices to watch are experian and the changed-rows subpopulations. Run in the
model-engine kernel after both builds finish.

In [1]:
import os, sys
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
sys.path.insert(0, os.getcwd())
from configs import DATA_DIR

TARGET     = 'final_DQ60_m24'
BUREAUS    = ['equifax', 'experian', 'transunion']
MODELS_DIR = os.path.join(DATA_DIR, 'models')
NEW_DIR    = os.path.join(DATA_DIR, 'new_normalized_and_processed')
VARIANTS   = ['npo_new', 'npo_old']

# variant -> where each bureau's TEST trade features live
def trade_dir(variant, bureau):
    if variant == 'npo_new' and bureau == 'experian':
        return os.path.join(NEW_DIR, 'experian_test', 'processed')
    v = 'new' if variant == 'npo_new' else 'old'
    return os.path.join(DATA_DIR, 'samples', f'{bureau}_test', f'processed_{v}')

pd.set_option('display.float_format', lambda x: f'{x:.5f}')
print('models dir:', MODELS_DIR)

models dir: /home/jag/payment-processor-research/payment_processing_research_data/models


In [2]:
# base frame: app (tagged with bureau) + target, test samples
app = pd.concat([
    pd.read_parquet(os.path.join(DATA_DIR, 'samples', f'{b}_test', 'app.parquet'),
                    columns=['ZEST_KEY', 'appDate']).assign(bureau=b)
    for b in BUREAUS], ignore_index=True)
tgt = pd.concat([
    pd.read_parquet(os.path.join(DATA_DIR, 'samples', f'{b}_test', 'target.parquet'),
                    columns=['ZEST_KEY', TARGET])
    for b in BUREAUS], ignore_index=True)
base = app.merge(tgt, on='ZEST_KEY', how='inner')
print('base:', base.shape)
print(base['bureau'].value_counts())

base: (1200000, 4)
bureau
equifax       400000
experian      400000
transunion    400000
Name: count, dtype: int64


In [3]:
# thin-file flag (date/count based, unaffected by the pattern changes)
TRADE_COLS = ['trade_months_since_oldest_account_opened__all_accounts',
              'trade_count__all_accounts']

def load_trade_cols(variant, cols):
    parts = []
    for b in BUREAUS:
        df = pd.read_parquet(trade_dir(variant, b), columns=cols)
        parts.append(df.reset_index())   # ZEST_KEY index -> column
    return pd.concat(parts, ignore_index=True)

base = base.merge(load_trade_cols('npo_new', TRADE_COLS), on='ZEST_KEY', how='left')
base['flg_thin_file'] = ((base[TRADE_COLS[0]] <= 6) | (base[TRADE_COLS[1]] <= 2))
print('thin file rate:', round(base['flg_thin_file'].mean(), 4))

thin file rate: 0.0551


In [4]:
# two probe features -- one percent, one number -- compared NEW vs OLD
PROBES = {
    'pct_changed': 'trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts',
    'num_changed': 'trade_max_number_of_DQ30_in_last_24_months__all_open_accounts',
}

# idempotent on re-runs
base = base.drop(columns=[c for k in PROBES for c in
                          (PROBES[k] + '__new', PROBES[k] + '__old', k)],
                 errors='ignore')

for flag, col in PROBES.items():
    a = load_trade_cols('npo_new', [col]).rename(columns={col: col + '__new'})
    o = load_trade_cols('npo_old', [col]).rename(columns={col: col + '__old'})
    base = base.merge(a, on='ZEST_KEY', how='left').merge(o, on='ZEST_KEY', how='left')
    missing = base[col + '__new'].isna() | base[col + '__old'].isna()
    base[flag] = (base[col + '__new'] != base[col + '__old']).astype('float64')
    base.loc[missing, flag] = np.nan
    print(f'{flag} ({col.split("__")[0]}):')
    print(base.groupby('bureau')[flag].mean().round(4).to_string(), '\n')

pct_changed (trade_mean_percent_of_DQ30_in_last_24_months):
bureau
equifax      0.07010
experian     0.06420
transunion   0.06590 

num_changed (trade_max_number_of_DQ30_in_last_24_months):
bureau
equifax      0.00000
experian     0.00270
transunion   0.00000 



In [5]:
# merge in both score sets
SCORE_COLS = {}

def load_scores(variant):
    p = os.path.join(MODELS_DIR, f'model_{variant}', 'test_scores.parquet')
    s = pd.read_parquet(p)
    if 'ZEST_KEY' not in s.columns:
        s = s.reset_index()
    assert 'ZEST_KEY' in s.columns, f'no ZEST_KEY in {p} (cols={list(s.columns)})'
    raw_cols = [c for c in s.columns if c != 'ZEST_KEY']
    s = s.rename(columns={c: f'{c}_{variant}' for c in raw_cols})
    suffixed = [f'{c}_{variant}' for c in raw_cols]
    pred = next((c for c in suffixed if any(k in c.lower() for k in ('score', 'pred', 'prob'))),
                suffixed[0])
    SCORE_COLS[variant] = pred
    print(f'{variant}: {len(s):,} rows | prediction -> {pred}')
    return s[['ZEST_KEY', pred]]

scored = base.copy()
for v in VARIANTS:
    scored = scored.merge(load_scores(v), on='ZEST_KEY', how='left')
print('\nscored:', scored.shape)

npo_new: 1,200,000 rows | prediction -> final_model_predictions_npo_new
npo_old: 1,200,000 rows | prediction -> final_model_predictions_npo_old

scored: (1200000, 15)


In [6]:
# AUC on every slice
def auc_row(df, label):
    y = df[TARGET]
    row = {'slice': label, 'n': int(len(df)), 'bad_rate': float(y.mean())}
    for v in VARIANTS:
        col = SCORE_COLS[v]
        m = y.notna() & df[col].notna()
        row[f'auc_{v}'] = roc_auc_score(y[m], df.loc[m, col]) if m.sum() and y[m].nunique() > 1 else np.nan
    return row

rows = [auc_row(scored, 'overall')]
for b in BUREAUS:
    rows.append(auc_row(scored[scored['bureau'] == b], b))
rows.append(auc_row(scored[scored['flg_thin_file'] == True], 'thin_file'))
rows.append(auc_row(scored[scored['pct_changed'] == 1],  'pct_probe_changed'))
rows.append(auc_row(scored[scored['num_changed'] == 1],  'num_probe_changed'))
rows.append(auc_row(scored[(scored['bureau'] == 'experian') & (scored['num_changed'] == 1)],
                    'experian_and_num_changed'))

auc = pd.DataFrame(rows).set_index('slice')
auc['new_minus_old'] = auc['auc_npo_new'] - auc['auc_npo_old']
auc.round(4)

,n,bad_rate,auc_npo_new,auc_npo_old,new_minus_old
slice,,,,,
overall,1200000,0.07100,0.72180,0.72180,-0.00010
equifax,400000,0.06980,0.72770,0.72750,0.00010
experian,400000,0.07810,0.72110,0.72120,-0.00010
transunion,400000,0.06490,0.71680,0.71650,0.00020
thin_file,66089,0.14320,0.62830,0.62770,0.00060
pct_probe_changed,79028,0.15410,0.67830,0.67580,0.00240
num_probe_changed,1064,0.13440,0.70060,0.70100,-0.00050
experian_and_num_changed,1064,0.13440,0.70060,0.70100,-0.00050


In [7]:
# score-level agreement
from scipy.stats import spearmanr

ca, cb = SCORE_COLS['npo_new'], SCORE_COLS['npo_old']
for slc, df in [('overall', scored),
                ('experian', scored[scored['bureau'] == 'experian']),
                ('experian_num_changed', scored[(scored['bureau'] == 'experian') & (scored['num_changed'] == 1)])]:
    m = df[ca].notna() & df[cb].notna()
    rho = spearmanr(df.loc[m, ca], df.loc[m, cb]).statistic
    mad = (df.loc[m, ca] - df.loc[m, cb]).abs().mean()
    print(f'npo_new vs npo_old [{slc:22}]: spearman={rho:.5f}  mean|diff|={mad:.6f}  n={m.sum():,}')

npo_new vs npo_old [overall               ]: spearman=0.99540  mean|diff|=0.005069  n=1,200,000
npo_new vs npo_old [experian              ]: spearman=0.99342  mean|diff|=0.005512  n=400,000
npo_new vs npo_old [experian_num_changed  ]: spearman=0.94345  mean|diff|=0.027173  n=1,064


In [8]:
# feature drift on the model's actual 288-feature FE matrix, by family
def load_fe(variant):
    return pd.read_parquet(os.path.join(MODELS_DIR, f'model_{variant}', 'test_fe_data.parquet'))

fe_new, fe_old = load_fe('npo_new'), load_fe('npo_old')
cols = [c for c in fe_new.columns if c in fe_old.columns]
idx  = fe_new.index.intersection(fe_old.index)
n, o = fe_new.loc[idx, cols], fe_old.loc[idx, cols]

rows = []
for c in cols:
    if n[c].dtype.kind not in 'fiub':
        continue
    af, bf = n[c].astype('float64'), o[c].astype('float64')
    neq = ~np.isclose(af, bf, equal_nan=True)
    if neq.any():
        absdiff = (af - bf).abs()
        rows.append({'col': c, 'n_changed': int(neq.sum()), 'pct_rows': 100 * neq.mean(),
                     'mad_changed': float(absdiff[neq].mean()),
                     'family': 'number' if 'number_of' in c else 'percent'})
d = pd.DataFrame(rows).sort_values('pct_rows', ascending=False).reset_index(drop=True)
print(f'{len(d)} of {len(cols)} columns differ (rows: {len(idx):,})')
print('\nby family:')
print(d.groupby('family').agg(n_cols=('col', 'count'), max_pct_rows=('pct_rows', 'max'),
                              mean_mad=('mad_changed', 'mean')).round(3))
print('\ntop 20 by pct_rows:')
print(d.head(20).to_string(index=False))

285 of 289 columns differ (rows: 1,200,000)

by family:
         n_cols  max_pct_rows  mean_mad
family                                 
number      128       0.18400   1.19400
percent     157       6.58600   0.24500

top 20 by pct_rows:
                                                                         col  n_changed  pct_rows  mad_changed  family
             trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts      79028   6.58567      0.02021 percent
              trade_max_percent_of_DQ30_in_last_24_months__all_open_accounts      70224   5.85200      0.07501 percent
            trade_mean_percent_of_DQ30_in_last_24_months__all_open_revolving      44532   3.71100      0.02864 percent
             trade_max_percent_of_DQ30_in_last_24_months__all_open_revolving      41321   3.44342      0.06562 percent
          trade_mean_percent_of_DQ30_in_last_24_months__all_open_charge_card      34894   2.90783      0.02903 percent
           trade_max_percent_of_DQ30_in_last_24_m